# Laboratório de Aprendizado de Máquina Supervisionado
### Aula IAML 05 — Churn, Preço de imóveis e Inadimplência em empréstimos

Rode as células **em ordem, de cima para baixo** (Ambiente de execução → Executar tudo).
Ao final, uma janela com o app vai abrir dentro do próprio Colab.

## 1. Instalar dependências

In [ ]:
%pip install -q scikit-learn==1.9.1
# Flask, pandas, numpy, matplotlib e seaborn já vêm instalados no Colab

## 2. Criar a pasta de templates

In [ ]:
!mkdir -p templates

## 3. Configuração: `config.py`
Descreve os 3 problemas (churn, imóveis, crédito), suas features e limites.

In [ ]:
%%writefile config.py
"""Configuração: os problemas de negócio, suas features e seus limites."""
from pathlib import Path

PASTA = Path(__file__).resolve().parent
DADOS = PASTA / "data"
MODELOS = PASTA / "models"
SEMENTE = 42  # fixa a aleatoriedade: todos obtêm os mesmos resultados


def num(padrao, minimo, maximo):
    return {"tipo": "numero", "padrao": padrao, "min": minimo, "max": maximo}


def cat(*opcoes):
    return {"tipo": "categoria", "padrao": opcoes[0], "opcoes": list(opcoes)}


PROBLEMAS = {
    "churn": {
        "titulo": "Churn de clientes",
        "tipo": "classificacao",
        "arquivo": "churn.csv",
        "alvo": "cancelou",
        "resultado": "Probabilidade de cancelamento",
        "features": {
            "idade": num(35, 18, 90),
            "meses_contrato": num(6, 0, 72),
            "tipo_contrato": cat("mensal", "anual", "bianual"),
            "pagamento": cat("boleto", "cartao", "debito_automatico", "pix"),
            "internet": cat("fibra", "dsl", "sem_internet"),
            "streaming": cat("nao", "sim"),
            "mensalidade": num(150, 10, 400),
            "chamados_suporte": num(3, 0, 30),
            "atrasos_pagamento": num(1, 0, 12),
        },
    },
    "imoveis": {
        "titulo": "Preço de imóveis",
        "tipo": "regressao",
        "arquivo": "imoveis.csv",
        "alvo": "preco",
        "resultado": "Preço estimado (R$)",
        "features": {
            "tipo": cat("apartamento", "casa"),
            "bairro": cat("Centro", "Jardins", "Vila Nova", "Beira-Mar", "Industrial"),
            "area_m2": num(80, 20, 800),
            "quartos": num(2, 1, 8),
            "banheiros": num(2, 1, 6),
            "vagas": num(1, 0, 6),
            "idade_imovel": num(10, 0, 100),
            "distancia_metro_km": num(1.5, 0, 30),
            "piscina": cat("nao", "sim"),
        },
    },
    # --- Parte 5 do exercício: problema de crédito adicionado ao sistema ---
    "credito": {
        "titulo": "Inadimplência em empréstimos",
        "tipo": "classificacao",
        "arquivo": "credito.csv",
        "alvo": "inadimplente",
        "resultado": "Probabilidade de inadimplência",
        "features": {
            "idade": num(40, 18, 80),
            "renda_mensal": num(4200, 1300, 60000),
            "tempo_emprego_anos": num(3.5, 0, 40),
            "score_credito": num(640, 300, 1000),
            "dividas_ativas": num(1, 0, 15),
            "possui_imovel": cat("nao", "sim"),
            "finalidade": cat("pessoal", "veiculo", "reforma", "educacao", "negocio"),
            "prazo_meses": num(36, 12, 60),
            "valor_emprestimo": num(15000, 1000, 200000),
        },
    },
}


## 4. Geração dos dados sintéticos: `gerar_dados.py`

In [ ]:
%%writefile gerar_dados.py
"""Gera dados sintéticos com padrões realistas de negócio. Uso: python gerar_dados.py"""
import numpy as np
import pandas as pd

from config import DADOS, SEMENTE

rng = np.random.default_rng(SEMENTE)


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def escolher(opcoes, pesos, n):
    return rng.choice(opcoes, n, p=pesos)


def apagar(df, coluna, fracao):
    """Simula valores ausentes, muito comuns em dados reais."""
    df.loc[rng.random(len(df)) < fracao, coluna] = np.nan


def gerar_churn(n=5000):
    d = pd.DataFrame({"id_cliente": [f"C{i:05d}" for i in range(1, n + 1)]})
    d["idade"] = np.clip(rng.normal(42, 13, n), 18, 85).round()
    d["meses_contrato"] = np.clip(rng.gamma(1.4, 17, n), 0, 72).round()
    d["tipo_contrato"] = escolher(["mensal", "anual", "bianual"], [.55, .25, .2], n)
    d["pagamento"] = escolher(["boleto", "cartao", "debito_automatico", "pix"], [.25, .3, .2, .25], n)
    d["internet"] = escolher(["fibra", "dsl", "sem_internet"], [.5, .35, .15], n)
    d["streaming"] = escolher(["sim", "nao"], [.4, .6], n)
    fibra = d.internet == "fibra"
    base = np.select([fibra, d.internet == "dsl"], [120, 80], 45)
    d["mensalidade"] = np.clip(base + 35 * (d.streaming == "sim") + rng.normal(0, 15, n), 20, 350).round(2)
    d["chamados_suporte"] = rng.poisson(1 + .8 * fibra)
    d["atrasos_pagamento"] = rng.poisson(.5 + .6 * (d.pagamento == "boleto"))
    z = (-1.9 + 1.3 * (d.tipo_contrato == "mensal") - .9 * (d.tipo_contrato == "bianual")
         - .045 * d.meses_contrato + .38 * d.chamados_suporte + .012 * (d.mensalidade - 100)
         + .35 * (d.pagamento == "boleto") - .45 * (d.pagamento == "debito_automatico")
         + .45 * d.atrasos_pagamento + .3 * fibra - .012 * (d.idade - 42) + rng.normal(0, .6, n))
    d["cancelou"] = (rng.random(n) < sigmoide(z)).astype(int)
    apagar(d, "idade", .03)
    apagar(d, "chamados_suporte", .02)
    return d


def gerar_imoveis(n=4000):
    d = pd.DataFrame({"id_imovel": [f"I{i:05d}" for i in range(1, n + 1)]})
    d["tipo"] = escolher(["apartamento", "casa"], [.65, .35], n)
    d["bairro"] = escolher(["Centro", "Jardins", "Vila Nova", "Beira-Mar", "Industrial"],
                            [.25, .18, .27, .12, .18], n)
    casa = d.tipo == "casa"
    d["area_m2"] = np.clip(np.where(casa, 140, 70) * rng.lognormal(0, .35, n), 25, 700).round()
    d["quartos"] = np.clip(np.round(d.area_m2 / 38 + rng.normal(0, .7, n)), 1, 7)
    d["banheiros"] = np.clip(np.round(d.quartos * .7 + rng.normal(0, .5, n)), 1, 6)
    d["vagas"] = np.clip(np.round(d.area_m2 / 70 + rng.normal(0, .6, n)), 0, 5)
    d["idade_imovel"] = rng.integers(0, 51, n)
    d["distancia_metro_km"] = np.clip(rng.exponential(2.2, n), .1, 20).round(1)
    nobre = d.bairro.isin(["Jardins", "Beira-Mar"])
    d["piscina"] = np.where(rng.random(n) < .08 + .25 * casa + .2 * nobre, "sim", "nao")
    m2 = d.bairro.map({"Centro": 7000, "Jardins": 11000, "Vila Nova": 5000,
                        "Beira-Mar": 13000, "Industrial": 3500})
    preco = (d.area_m2 * m2 * (1 - .007 * d.idade_imovel) * (1 + .07 * d.vagas)
             * np.exp(-.05 * d.distancia_metro_km) * np.where(d.piscina == "sim", 1.12, 1)
             * np.where(casa, .92, 1) * rng.lognormal(0, .12, n))
    d["preco"] = (preco / 1000).round() * 1000
    apagar(d, "vagas", .02)
    apagar(d, "distancia_metro_km", .03)
    return d


def gerar_credito(n=6000):
    """Dados do exercício: inadimplência em empréstimos."""
    d = pd.DataFrame({"id_contrato": [f"E{i:05d}" for i in range(1, n + 1)]})
    d["idade"] = np.clip(rng.normal(40, 12, n), 18, 80).round()
    d["renda_mensal"] = np.clip(4200 * rng.lognormal(0, .6, n), 1300, 60000).round(-1)
    d["tempo_emprego_anos"] = np.clip(rng.gamma(1.6, 3.5, n), 0, 40).round(1)
    d["score_credito"] = np.clip(rng.normal(640, 110, n), 300, 1000).round()
    d["dividas_ativas"] = rng.poisson(1.1, n)
    d["possui_imovel"] = escolher(["sim", "nao"], [.4, .6], n)
    d["finalidade"] = escolher(["pessoal", "veiculo", "reforma", "educacao", "negocio"],
                                [.35, .25, .15, .1, .15], n)
    d["prazo_meses"] = escolher([12, 24, 36, 48, 60], [.15, .3, .25, .15, .15], n)
    d["valor_emprestimo"] = np.clip(d.renda_mensal * rng.uniform(.5, 6, n), 1000, 200000).round(-2)
    comprometimento = d.valor_emprestimo * 1.33 / d.prazo_meses / d.renda_mensal
    z = (-2.2 + 2.8 * np.clip(comprometimento, 0, 1.5) - .009 * (d.score_credito - 640)
         + .4 * d.dividas_ativas - .06 * d.tempo_emprego_anos - .35 * (d.possui_imovel == "sim")
         + .45 * (d.finalidade == "negocio") - .015 * (d.idade - 40) + rng.normal(0, .5, n))
    d["inadimplente"] = (rng.random(n) < sigmoide(z)).astype(int)
    apagar(d, "renda_mensal", .04)
    apagar(d, "tempo_emprego_anos", .03)
    return d


def main():
    DADOS.mkdir(exist_ok=True)
    for nome, gerar in [("churn", gerar_churn), ("imoveis", gerar_imoveis), ("credito", gerar_credito)]:
        df = gerar()
        df.to_csv(DADOS / f"{nome}.csv", index=False)
        print(f"data/{nome}.csv: {len(df)} linhas")


if __name__ == "__main__":
    main()


## 5. Treino e comparação de modelos: `treinar.py`

In [ ]:
%%writefile treinar.py
"""Compara algoritmos, avalia no teste e salva os modelos. Uso: python treinar.py"""
import json

import joblib
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                               RandomForestClassifier, RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
                              precision_score, r2_score, recall_score, roc_auc_score,
                              root_mean_squared_error)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from config import DADOS, MODELOS, PROBLEMAS, SEMENTE


def candidatos(tipo):
    if tipo == "classificacao":
        return {
            "Regressão Logística": LogisticRegression(max_iter=1000),
            "Árvore de Decisão": DecisionTreeClassifier(max_depth=5, random_state=SEMENTE),
            "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=10,
                                                     min_samples_leaf=5, random_state=SEMENTE),
            "Gradient Boosting": HistGradientBoostingClassifier(learning_rate=.05, random_state=SEMENTE),
        }
    return {
        "Regressão Linear": LinearRegression(),
        "Árvore de Decisão": DecisionTreeRegressor(max_depth=8, random_state=SEMENTE),
        "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=14,
                                                min_samples_leaf=3, random_state=SEMENTE),
        "Gradient Boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=.05,
                                                            random_state=SEMENTE),
    }


def criar_pipeline(problema, modelo):
    features = problema["features"]
    numericas = [f for f in features if features[f]["tipo"] == "numero"]
    categoricas = [f for f in features if features[f]["tipo"] == "categoria"]
    preprocessamento = ColumnTransformer([
        ("numericas", Pipeline([("preencher", SimpleImputer(strategy="median")),
                                 ("padronizar", StandardScaler())]), numericas),
        ("categoricas", Pipeline([("preencher", SimpleImputer(strategy="most_frequent")),
                                   ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), categoricas),
    ])
    return Pipeline([("preprocessamento", preprocessamento), ("modelo", modelo)])


def dividir(problema):
    df = pd.read_csv(DADOS / problema["arquivo"])
    X = df[list(problema["features"])]  # só as features: o id e o alvo ficam de fora
    y = df[problema["alvo"]]
    estratificar = y if problema["tipo"] == "classificacao" else None
    return train_test_split(X, y, test_size=.2, random_state=SEMENTE, stratify=estratificar)


def avaliar(problema, modelo, X_teste, y_teste):
    if problema["tipo"] == "classificacao":
        prob = modelo.predict_proba(X_teste)[:, 1]
        prev = (prob >= .5).astype(int)
        return {"acuracia": accuracy_score(y_teste, prev), "precisao": precision_score(y_teste, prev),
                "recall": recall_score(y_teste, prev), "f1": f1_score(y_teste, prev),
                "roc_auc": roc_auc_score(y_teste, prob),
                "matriz_confusao": confusion_matrix(y_teste, prev).tolist()}
    prev = modelo.predict(X_teste)
    return {"mae": mean_absolute_error(y_teste, prev), "rmse": root_mean_squared_error(y_teste, prev),
            "r2": r2_score(y_teste, prev)}


def treinar(nome, problema):
    print(f"\n=== {problema['titulo']} ===")
    X_treino, X_teste, y_treino, y_teste = dividir(problema)
    classificacao = problema["tipo"] == "classificacao"
    metrica = "roc_auc" if classificacao else "neg_root_mean_squared_error"

    comparacao = {}
    for algoritmo, modelo in candidatos(problema["tipo"]).items():
        cv = cross_validate(criar_pipeline(problema, modelo), X_treino, y_treino, cv=5, scoring=metrica)
        notas = cv["test_score"] if classificacao else -cv["test_score"]  # RMSE vem negativo
        comparacao[algoritmo] = {"media": notas.mean(), "desvio": notas.std()}
        print(f"  {algoritmo:<20} {notas.mean():.4f} ± {notas.std():.4f}")

    escolhido = (max if classificacao else min)(comparacao, key=lambda a: comparacao[a]["media"])
    modelo = criar_pipeline(problema, candidatos(problema["tipo"])[escolhido]).fit(X_treino, y_treino)
    teste = avaliar(problema, modelo, X_teste, y_teste)
    print(f"  Escolhido: {escolhido} | teste: {teste}")

    joblib.dump(modelo, MODELOS / f"{nome}.joblib", compress=3)
    return {"modelo": escolhido, "metrica": "ROC AUC" if classificacao else "RMSE",
            "validacao_cruzada": comparacao, "teste": teste, "sklearn": sklearn.__version__}


def main():
    MODELOS.mkdir(exist_ok=True)
    metricas = {nome: treinar(nome, problema) for nome, problema in PROBLEMAS.items()}
    (MODELOS / "metricas.json").write_text(json.dumps(metricas, indent=2, ensure_ascii=False),
                                            encoding="utf-8")


if __name__ == "__main__":
    main()


## 6. API: `app.py`

In [ ]:
%%writefile app.py
"""API e interface web. Uso: python app.py -> http://localhost:5000"""
import json
import os

import joblib
import pandas as pd
from flask import Flask, jsonify, render_template, request

from config import DADOS, MODELOS, PROBLEMAS

if not (MODELOS / "metricas.json").exists():  # primeira execução: gera os dados e treina
    import gerar_dados
    import treinar

    if not (DADOS / "churn.csv").exists():
        gerar_dados.main()
    treinar.main()

METRICAS = json.loads((MODELOS / "metricas.json").read_text(encoding="utf-8"))
MODELOS_CARREGADOS = {nome: joblib.load(MODELOS / f"{nome}.joblib") for nome in PROBLEMAS}

app = Flask(__name__)
app.json.ensure_ascii = False
app.json.sort_keys = False


def validar(problema, dados):
    """Nunca confie na entrada: confere tipos, limites e opções de cada campo."""
    linha, erros = {}, []
    for campo, regra in problema["features"].items():
        valor = dados.get(campo)
        if valor in (None, ""):  # vazio é permitido: o pipeline preenche
            linha[campo] = None
        elif regra["tipo"] == "categoria":
            if valor not in regra["opcoes"]:
                erros.append(f"{campo} deve ser um de: {', '.join(regra['opcoes'])}")
            linha[campo] = valor
        else:
            try:
                linha[campo] = float(valor)
                if not regra["min"] <= linha[campo] <= regra["max"]:
                    erros.append(f"{campo} deve estar entre {regra['min']} e {regra['max']}")
            except (TypeError, ValueError):
                erros.append(f"{campo} deve ser numérico")
    return linha, erros


@app.get("/")
def pagina():
    return render_template("index.html")


@app.get("/api/problemas")
def problemas():
    return jsonify({nome: {**p, "metricas": METRICAS[nome]} for nome, p in PROBLEMAS.items()})


@app.post("/api/prever/<nome>")
def prever(nome):
    if nome not in PROBLEMAS:
        return jsonify(erro="Problema não encontrado"), 404
    problema = PROBLEMAS[nome]
    linha, erros = validar(problema, request.get_json(silent=True) or {})
    if erros:
        return jsonify(erro="; ".join(erros)), 400

    X = pd.DataFrame([linha], columns=list(problema["features"])).astype(
        {c: float for c, r in problema["features"].items() if r["tipo"] == "numero"})
    modelo = MODELOS_CARREGADOS[nome]
    if problema["tipo"] == "classificacao":
        return jsonify(probabilidade=float(modelo.predict_proba(X)[0, 1]))
    return jsonify(valor=float(modelo.predict(X)[0]))


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", 5000)))


## 7. Interface web: `templates/index.html`

In [ ]:
%%writefile templates/index.html
<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Laboratório de ML Supervisionado</title>
<style>
  :root { --fundo: #f4f5f8; --cartao: #fff; --texto: #1d2330; --suave: #5b6475; --borda: #e1e4ea; --cor: #3b5bdb; --ruim: #c92a2a; --bom: #2b8a3e; }
  @media (prefers-color-scheme: dark) {
    :root { --fundo: #12151c; --cartao: #1b1f29; --texto: #e6e9ef; --suave: #9aa3b5; --borda: #2c3240; --cor: #748ffc; --ruim: #ff8787; --bom: #69db7c; }
  }
  * { box-sizing: border-box; }
  body { margin: 0; font-family: system-ui, sans-serif; background: var(--fundo); color: var(--texto); }
  header { background: #1d2330; color: #fff; padding: 28px 16px; }
  header h1 { margin: 0 0 6px; font-size: 1.7rem; }
  header p { margin: 0; opacity: .8; }
  main { max-width: 1100px; margin: 0 auto; padding: 20px 16px; }
  nav button { border: 1px solid var(--borda); background: var(--cartao); color: var(--texto); padding: 9px 16px; border-radius: 99px; font: inherit; font-weight: 600; cursor: pointer; margin: 0 6px 16px 0; }
  nav button.ativo { background: var(--cor); border-color: var(--cor); color: #fff; }
  .grade { display: grid; grid-template-columns: 1fr 1fr; gap: 18px; align-items: start; }
  @media (max-width: 800px) { .grade { grid-template-columns: 1fr; } }
  section { background: var(--cartao); border: 1px solid var(--borda); border-radius: 12px; padding: 18px; margin-bottom: 18px; }
  h2 { margin: 0 0 14px; font-size: 1.05rem; }
  form { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
  label { font-size: .85rem; font-weight: 600; }
  input, select { width: 100%; margin-top: 4px; padding: 8px; border: 1px solid var(--borda); border-radius: 8px; background: var(--fundo); color: var(--texto); font: inherit; }
  form button { grid-column: 1 / -1; background: var(--cor); color: #fff; border: 0; border-radius: 8px; padding: 11px; font: inherit; font-weight: 600; cursor: pointer; }
  .valor { font-size: 2.2rem; font-weight: 700; margin: 4px 0; }
  .barra { height: 10px; background: var(--borda); border-radius: 99px; overflow: hidden; }
  .barra span { display: block; height: 100%; background: var(--cor); }
  .suave { color: var(--suave); font-size: .85rem; }
  table { width: 100%; border-collapse: collapse; font-size: .9rem; }
  td { padding: 7px 4px; border-bottom: 1px solid var(--borda); }
  td:last-child { text-align: right; }
  .escolhido { font-weight: 700; color: var(--cor); }
</style>
</head>
<body>
<header>
  <h1>Laboratório de Aprendizado Supervisionado</h1>
  <p>Modelos treinados com scikit-learn e servidos por uma API Flask.</p>
</header>
<main>
  <nav id="abas"></nav>
  <div class="grade">
    <section>
      <h2>Dados de entrada</h2>
      <form id="formulario"></form>
    </section>
    <div>
      <section><h2>Previsão</h2><div id="resultado" class="suave">Preencha os dados e clique em Prever.</div></section>
      <section><h2>Avaliação do modelo</h2><div id="metricas"></div></section>
    </div>
  </div>
</main>
<script>
let problemas = {}, atual = null;
const fmt = (v, casas = 3) => v.toLocaleString("pt-BR", { maximumFractionDigits: casas });
const real = (v) => v.toLocaleString("pt-BR", { style: "currency", currency: "BRL", maximumFractionDigits: 0 });
const pct = (v) => fmt(v * 100, 1) + "%";

async function iniciar() {
  problemas = await (await fetch("/api/problemas")).json();
  const abas = document.getElementById("abas");
  for (const nome in problemas) {
    const botao = document.createElement("button");
    botao.textContent = problemas[nome].titulo;
    botao.onclick = () => selecionar(nome);
    abas.appendChild(botao);
  }
  selecionar(Object.keys(problemas)[0]);
}

function selecionar(nome) {
  atual = nome;
  const p = problemas[nome];
  document.querySelectorAll("nav button").forEach((b) => b.classList.toggle("ativo", b.textContent === p.titulo));
  let html = "";
  for (const [campo, r] of Object.entries(p.features)) {
    const entrada = r.tipo === "categoria"
      ? `<select name="${campo}">${r.opcoes.map((o) => `<option>${o}</option>`).join("")}</select>`
      : `<input name="${campo}" type="number" step="any" min="${r.min}" max="${r.max}" value="${r.padrao}">`;
    html += `<label>${campo.replaceAll("_", " ")}${entrada}</label>`;
  }
  const formulario = document.getElementById("formulario");
  formulario.innerHTML = html + "<button>Prever</button>";
  formulario.onsubmit = prever;
  document.getElementById("resultado").innerHTML = "Preencha os dados e clique em Prever.";
  mostrarMetricas(p);
}

async function prever(evento) {
  evento.preventDefault();
  const dados = Object.fromEntries(new FormData(evento.target));
  const resposta = await fetch(`/api/prever/${atual}`, {
    method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(dados),
  });
  const r = await resposta.json();
  const p = problemas[atual];
  let html = `<div class="suave">${p.resultado}</div>`;
  if (!resposta.ok) html = `<p style="color: var(--ruim)">${r.erro}</p>`;
  else if (r.valor !== undefined) html += `<div class="valor">${real(r.valor)}</div>`;
  else {
    const cor = r.probabilidade >= 0.5 ? "var(--ruim)" : "var(--bom)";
    html += `<div class="valor" style="color:${cor}">${pct(r.probabilidade)}</div>
      <div class="barra"><span style="width:${r.probabilidade * 100}%"></span></div>`;
  }
  document.getElementById("resultado").innerHTML = html;
}

function mostrarMetricas(p) {
  const m = p.metricas, regressao = p.tipo === "regressao";
  let html = `<p class="suave">Validação cruzada (5 folds) — ${m.metrica}, ${regressao ? "menor" : "maior"} é melhor</p><table>`;
  for (const [algoritmo, r] of Object.entries(m.validacao_cruzada)) {
    const valor = regressao ? `${real(r.media)} ± ${real(r.desvio)}` : `${fmt(r.media)} ± ${fmt(r.desvio)}`;
    html += `<tr class="${algoritmo === m.modelo ? "escolhido" : ""}"><td>${algoritmo}</td><td>${valor}</td></tr>`;
  }
  const t = m.teste;
  const teste = regressao
    ? { MAE: real(t.mae), RMSE: real(t.rmse), "R²": fmt(t.r2) }
    : { Acurácia: pct(t.acuracia), Precisão: pct(t.precisao), Recall: pct(t.recall), F1: fmt(t.f1), "ROC AUC": fmt(t.roc_auc) };
  html += `</table><p class="suave">Conjunto de teste — modelo escolhido: <b>${m.modelo}</b></p><table>`;
  for (const [nome, valor] of Object.entries(teste)) html += `<tr><td>${nome}</td><td>${valor}</td></tr>`;
  if (t.matriz_confusao) {
    const [[vn, fp], [fn, vp]] = t.matriz_confusao;
    html += `<tr><td>Matriz de confusão (VN, FP, FN, VP)</td><td>${vn}, ${fp}, ${fn}, ${vp}</td></tr>`;
  }
  document.getElementById("metricas").innerHTML = html + "</table>";
}

iniciar();
</script>
</body>
</html>


## 8. Gerar os dados

In [ ]:
%run gerar_dados.py

## 9. Treinar os modelos
Compara os algoritmos com validação cruzada, avalia no teste e salva os 3 modelos em `models/`.

In [ ]:
%run treinar.py

## 10. Abrir a interface web dentro do Colab

In [ ]:
import threading
from google.colab import output
from app import app

threading.Thread(target=lambda: app.run(port=5000), daemon=True).start()
output.serve_kernel_port_as_window(5000)

---
**Pronto!** A janela acima mostra o laboratório com 3 abas: *Churn de clientes*, *Preço de imóveis* e *Inadimplência em empréstimos*.

Se quiser testar a API diretamente por código (sem abrir a janela), rode em uma nova célula:
```python
import requests
r = requests.post("http://localhost:5000/api/prever/credito", json={
    "idade": 24, "renda_mensal": 1800, "tempo_emprego_anos": 0.5, "score_credito": 420,
    "dividas_ativas": 5, "possui_imovel": "nao", "finalidade": "negocio",
    "prazo_meses": 60, "valor_emprestimo": 9000
})
print(r.json())
```